## Batch Evaluation

In [3]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.9 MB/s eta 0:00:00:00:0100:01


In [16]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import glob
import re
import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

# =========================================================
# CONFIG
# =========================================================

EMBEDDINGS_DIR = "/kaggle/input/datasets/iharshsinha/vr-c-finaloutput/memberC_outputs"
METADATA_CSV = "/kaggle/input/datasets/iharshsinha/vr-c-finaloutput/memberC_outputs/metadata_with_embeddings.csv"
K_VALUES       = [5, 10, 15]

# =========================================================
# LOAD METADATA ONCE
# =========================================================

metadata   = pd.read_csv(METADATA_CSV)
gallery_df = metadata[metadata["split"] == "gallery"].reset_index(drop=True)
query_df   = metadata[metadata["split"] == "query"].reset_index(drop=True)

gallery_indices = gallery_df["embedding_index"].values

item_to_gallery_rows: dict[str, set] = {}
for row_idx, item_id in enumerate(gallery_df["item_id"]):
    item_to_gallery_rows.setdefault(item_id, set()).add(row_idx)

print(f"Metadata loaded — {len(query_df)} queries, {len(gallery_df)} gallery items\n")

# =========================================================
# FILE CLASSIFICATION
# Skip _std.npy (precomputed std arrays, not embeddings)
# Separate averaged files from per-seed files
# =========================================================

all_npy = sorted(glob.glob(os.path.join(EMBEDDINGS_DIR, "*.npy")))

std_files      = [f for f in all_npy if f.endswith("_std.npy")]
seed_files     = [f for f in all_npy if re.search(r"_seed\d+\.npy$", f)]
averaged_files = [f for f in all_npy if f not in std_files and f not in seed_files]

print(f"Found {len(all_npy)} total .npy files:")
print(f"  {len(averaged_files)} averaged  (main eval)")
print(f"  {len(seed_files)}  per-seed  (for mean ± std)")
print(f"  {len(std_files)}  std files  (SKIPPED — not embeddings)")
print()

# =========================================================
# METRIC FUNCTIONS
# =========================================================

def recall_at_k(relevant_set, retrieved, k):
    return int(any(r in relevant_set for r in retrieved[:k]))

def average_precision_at_k(relevant_set, retrieved, k):
    if not relevant_set:
        return 0.0
    score, hits = 0.0, 0
    for i, item in enumerate(retrieved[:k]):
        if item in relevant_set:
            hits  += 1
            score += hits / (i + 1)
    return score / min(len(relevant_set), k)

def dcg_at_k(relevant_set, retrieved, k):
    return sum(1.0 / np.log2(i + 2)
               for i, item in enumerate(retrieved[:k])
               if item in relevant_set)

def ndcg_at_k(relevant_set, retrieved, k):
    n_ideal = min(len(relevant_set), k)
    if n_ideal == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 2) for i in range(n_ideal))
    return dcg_at_k(relevant_set, retrieved, k) / ideal_dcg

# =========================================================
# EVALUATE ONE .npy FILE
# =========================================================

def evaluate(npy_path: str) -> dict:
    embeddings = np.load(npy_path).astype("float32")
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.where(norms == 0, 1, norms)

    gallery_embeddings = embeddings[gallery_indices].astype("float32")

    dim   = gallery_embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch       = 128
    index.add(gallery_embeddings)

    MAX_K   = max(K_VALUES)
    recalls = {k: [] for k in K_VALUES}
    aps     = {k: [] for k in K_VALUES}
    ndcgs   = {k: [] for k in K_VALUES}

    for i in range(len(query_df)):
        row          = query_df.iloc[i]
        relevant_set = item_to_gallery_rows.get(row["item_id"], set())
        query_vec    = embeddings[row["embedding_index"]].reshape(1, -1)

        _, indices = index.search(query_vec, MAX_K)
        retrieved  = [int(idx) for idx in indices[0] if idx >= 0]

        for k in K_VALUES:
            recalls[k].append(recall_at_k(relevant_set, retrieved, k))
            aps[k].append(average_precision_at_k(relevant_set, retrieved, k))
            ndcgs[k].append(ndcg_at_k(relevant_set, retrieved, k))

    return {
        k: {
            "Recall": np.mean(recalls[k]),
            "mAP":    np.mean(aps[k]),
            "NDCG":   np.mean(ndcgs[k]),
        }
        for k in K_VALUES
    }

# =========================================================
# ABLATION LABEL EXTRACTION
# e.g. "embeddings_ablationC_07.npy"   -> "C_07"
#      "embeddings_ablationA_seed571.npy" -> "A"
# =========================================================

def ablation_label(path: str) -> str:
    fname = os.path.basename(path)
    m = re.search(r"ablation([ABC](?:_\d+)?)", fname)
    return m.group(1) if m else fname

# =========================================================
# STEP 1 — EVALUATE AVERAGED FILES (primary results)
# =========================================================

print("=" * 60)
print("STEP 1: AVERAGED EMBEDDINGS (primary results)")
print("=" * 60)

averaged_results = {}   # label -> {k -> metrics}

for path in averaged_files:
    label = ablation_label(path)
    print(f"\nEvaluating Ablation {label}  [{os.path.basename(path)}]")
    results = evaluate(path)
    averaged_results[label] = results

    for k in K_VALUES:
        r = results[k]
        print(f"  @{k:<2}  Recall={r['Recall']:.4f}  mAP={r['mAP']:.4f}  NDCG={r['NDCG']:.4f}")

# =========================================================
# STEP 2 — EVALUATE PER-SEED FILES (for ± std)
# =========================================================

print("\n" + "=" * 60)
print("STEP 2: PER-SEED EMBEDDINGS (mean ± std)")
print("=" * 60)

# Group seed files by ablation condition
seed_groups: dict[str, list] = defaultdict(list)
for path in seed_files:
    label = ablation_label(path)
    seed_groups[label].append(path)

seed_results = {}   # label -> {k -> {"mean": x, "std": x} per metric}

for label, paths in sorted(seed_groups.items()):
    print(f"\nAblation {label}  ({len(paths)} seeds)")
    per_seed = []
    for path in paths:
        print(f"  seed: {os.path.basename(path)}")
        per_seed.append(evaluate(path))

    # Aggregate across seeds
    agg = {}
    for k in K_VALUES:
        recalls = [s[k]["Recall"] for s in per_seed]
        maps    = [s[k]["mAP"]    for s in per_seed]
        ndcgs   = [s[k]["NDCG"]   for s in per_seed]
        agg[k]  = {
            "Recall_mean": np.mean(recalls), "Recall_std": np.std(recalls),
            "mAP_mean":    np.mean(maps),    "mAP_std":    np.std(maps),
            "NDCG_mean":   np.mean(ndcgs),   "NDCG_std":   np.std(ndcgs),
        }
        print(f"  @{k:<2}  "
              f"Recall={agg[k]['Recall_mean']:.4f}±{agg[k]['Recall_std']:.4f}  "
              f"mAP={agg[k]['mAP_mean']:.4f}±{agg[k]['mAP_std']:.4f}  "
              f"NDCG={agg[k]['NDCG_mean']:.4f}±{agg[k]['NDCG_std']:.4f}")
    seed_results[label] = agg

# =========================================================
# FINAL REPORT TABLE  (matches project rubric format)
# =========================================================

print("\n" + "=" * 70)
print("FINAL REPORT TABLE  (mean ± std from seeds | averaged embedding result)")
print("=" * 70)

ABLATION_META = {
    "A":    "α=1.0 | frozen CLIP | vision only",
    "B_07": "α=0.7 | frozen CLIP | +captions",
    "B_05": "α=0.5 | frozen CLIP | +captions",
    "C_07": "α=0.7 | fine-tuned CLIP | +captions",
    "C_05": "α=0.5 | fine-tuned CLIP | +captions",
}

rows = []
for label in sorted(set(list(averaged_results.keys()) + list(seed_results.keys()))):
    desc = ABLATION_META.get(label, label)
    for k in K_VALUES:
        row = {"Ablation": label, "Config": desc, "K": k}
        if label in averaged_results:
            r = averaged_results[label][k]
            row["Recall (avg)"] = f"{r['Recall']:.4f}"
            row["mAP (avg)"]    = f"{r['mAP']:.4f}"
            row["NDCG (avg)"]   = f"{r['NDCG']:.4f}"
        if label in seed_results:
            s = seed_results[label][k]
            row["Recall (±std)"] = f"{s['Recall_mean']:.4f}±{s['Recall_std']:.4f}"
            row["mAP (±std)"]    = f"{s['mAP_mean']:.4f}±{s['mAP_std']:.4f}"
            row["NDCG (±std)"]   = f"{s['NDCG_mean']:.4f}±{s['NDCG_std']:.4f}"
        rows.append(row)

report_df = pd.DataFrame(rows)
print(report_df.to_string(index=False))

report_df.to_csv("/kaggle/working/eval_results_full.csv", index=False)
print("\nSaved to /kaggle/working/eval_results_full.csv")

Metadata loaded — 1330 queries, 15144 gallery items

Found 30 total .npy files:
  5 averaged  (main eval)
  20  per-seed  (for mean ± std)
  5  std files  (SKIPPED — not embeddings)

STEP 1: AVERAGED EMBEDDINGS (primary results)

Evaluating Ablation A  [embeddings_ablationA.npy]
  @5   Recall=0.4248  mAP=0.1394  NDCG=0.1932
  @10  Recall=0.4842  mAP=0.1306  NDCG=0.1931
  @15  Recall=0.5143  mAP=0.1311  NDCG=0.1983

Evaluating Ablation B_05  [embeddings_ablationB_05.npy]
  @5   Recall=0.3910  mAP=0.1338  NDCG=0.1829
  @10  Recall=0.4692  mAP=0.1306  NDCG=0.1904
  @15  Recall=0.5090  mAP=0.1321  NDCG=0.1977

Evaluating Ablation B_07  [embeddings_ablationB_07.npy]
  @5   Recall=0.4489  mAP=0.1621  NDCG=0.2186
  @10  Recall=0.5218  mAP=0.1566  NDCG=0.2239
  @15  Recall=0.5617  mAP=0.1586  NDCG=0.2333

Evaluating Ablation C_05  [embeddings_ablationC_05.npy]
  @5   Recall=0.7226  mAP=0.3464  NDCG=0.4278
  @10  Recall=0.7865  mAP=0.3439  NDCG=0.4424
  @15  Recall=0.8286  mAP=0.3479  NDCG=0.45

In [15]:
import os
import glob

BASE = "/kaggle/input/datasets/iharshsinha/vr-c-finaloutput"

# Walk the full directory tree
for root, dirs, files in os.walk(BASE):
    npy_files = [f for f in files if f.endswith(".npy")]
    if npy_files:
        print(f"\nFolder: {root}")
        for f in npy_files:
            print(f"  {f}")


Folder: /kaggle/input/datasets/iharshsinha/vr-c-finaloutput/memberC_outputs
  embeddings_ablationC_07_seed571.npy
  embeddings_ablationB_07_seed129.npy
  embeddings_ablationB_07_seed571.npy
  embeddings_ablationC_07_seed591.npy
  embeddings_ablationC_05_std.npy
  embeddings_ablationA_seed571.npy
  embeddings_ablationB_05_std.npy
  embeddings_ablationC_07_std.npy
  embeddings_ablationC_07.npy
  embeddings_ablationB_05_seed129.npy
  embeddings_ablationA.npy
  embeddings_ablationC_05_seed571.npy
  embeddings_ablationC_05_seed126.npy
  embeddings_ablationA_seed129.npy
  embeddings_ablationC_07_seed126.npy
  embeddings_ablationC_07_seed129.npy
  embeddings_ablationB_05.npy
  embeddings_ablationB_05_seed571.npy
  embeddings_ablationC_05.npy
  embeddings_ablationA_seed591.npy
  embeddings_ablationB_05_seed591.npy
  embeddings_ablationB_07.npy
  embeddings_ablationC_05_seed591.npy
  embeddings_ablationB_07_seed591.npy
  embeddings_ablationA_seed126.npy
  embeddings_ablationB_07_std.npy
  embe